# 引入套件

In [1]:
import os
import pandas as pd
import numpy as np
from finlab.dataframe import FinlabDataFrame
from finlab.backtest import sim
from finlab import data
import talib
import finlab
import logging
import openpyxl
from scipy.stats import linregress
from dotenv import load_dotenv
# 載入環境變數
load_dotenv()
finlab.login()
# 使用環境變數
# finlab.login(os.getenv('FINLAB_API_KEY'))
# 設置 finlab 數據存儲路徑


已登入（使用快取憑證）。


# 下載ETF資料

In [ ]:
"""
此程式整合了三種基金與ETF持股資料的爬取功能，並將資料儲存至各自獨立的CSV檔案中。
採用多執行緒 (Multi-threading) 方式並行抓取，以提升執行效率。

1. [每日] 主動式 ETF 持股明細 -> all_etf_holdings.csv (並行 Selenium)
2. [每月] 投信基金前十大投資明細 -> sitca_fund_holdings.csv (並行 Requests)
3. [每季] 投信基金占淨值1%以上投資明細 -> sitca_fund_holdings_over_1_percent_quarterly.csv (並行 Requests)

程式會自動讀取對應的資料檔案，判斷最新時間點，並進行增量更新。
"""

# doc: =============================================================================
# doc: 匯入所有必要的函式庫
# doc: =============================================================================
import pandas as pd
import time
import os
import re
import requests
from bs4 import BeautifulSoup
from datetime import datetime
from io import StringIO
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
# [修改] 引入 Selenium 的例外處理
from selenium.common.exceptions import TimeoutException, NoSuchElementException, WebDriverException
# NEW: 匯入 concurrent.futures 模組
from concurrent.futures import ThreadPoolExecutor, as_completed
import logging  # [修改] 增加 logging 模組
# doc: =============================================================================
# doc: 全域設定
# doc: =============================================================================
# SITCA 爬蟲的並行數量 (避免對伺服器造成過大壓力)
MAX_WORKERS_SITCA = 8

# doc: =============================================================================
# doc: (爬蟲核心 1/3) 每日主動式 ETF
# doc: =============================================================================


def get_etf_holdings_selenium(stock_id: str) -> tuple[pd.DataFrame | None, str | None]:
    """
    doc: 使用 Selenium 抓取單一 ETF 的持股明細及更新日期。
    [修改] 增加完整的 try...except...finally 錯誤處理機制。
    
    Args:
        stock_id (str): ETF 的代號。

    Returns:
        tuple[pd.DataFrame | None, str | None]: 包含 (DataFrame, 更新日期字串) 的元組，失敗則返回 (None, None)。
    """
    url = f"https://www.pocket.tw/etf/tw/{stock_id}/fundholding"
    service = Service()
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')  # [修改] 建議保持 headless 模式
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--log-level=3')
    options.add_experimental_option('excludeSwitches', ['enable-logging'])

    driver = None
    try:
        driver = webdriver.Chrome(service=service, options=options)
        driver.get(url)

        wait = WebDriverWait(driver, 20)  # 等待時間設為 20 秒

        # 1. 等待持股表格出現
        wait.until(EC.presence_of_element_located(
            (By.CSS_SELECTOR, ".cm-table__table tbody tr")))

        # 2. 等待日期資訊出現
        wait.until(EC.presence_of_element_located(
            (By.CSS_SELECTOR, ".fundholding__date")))

        time.sleep(1)  # 最終等待渲染

        # 3. 抓取日期
        date_elements = driver.find_elements(
            By.CSS_SELECTOR, ".fundholding__date")
        update_date = date_elements[0].text.replace(
            '資料日期：', '').strip() if date_elements else None

        if not update_date:
            logging.warning(f"爬取 {stock_id} ({url}) 時找不到日期元素。")
            return None, None

        # 4. 抓取表格
        tables = pd.read_html(StringIO(driver.page_source))

        fund_holdings_df = None
        for table in tables:
            if len(table.columns) >= 2 and isinstance(table.columns[0], str) and isinstance(table.columns[1], str):
                if '代號' in table.columns[0] and '名稱' in table.columns[1]:
                    fund_holdings_df = table
                    break

        if fund_holdings_df is None:
            logging.warning(f"錯誤：在 {stock_id} 頁面中找不到持股表格。")
            return None, update_date

        df = fund_holdings_df.iloc[:, 0:4].copy()
        df.columns = ['代號', '名稱', '權重', '持有數']

        df['權重'] = pd.to_numeric(df['權重'].astype(str).str.replace(
            '%', '', regex=False), errors='coerce')
        df['持有數'] = pd.to_numeric(df['持有數'].astype(
            str).str.replace(',', '', regex=False), errors='coerce')
        df.dropna(subset=['代號', '名稱', '權重', '持有數'], inplace=True)

        df['持有數'] = (df['持有數'] / 1000).astype(int)
        df['單位'] = '張'

        return df, update_date

    # [修改] 增加更詳細的錯誤捕捉
    except (TimeoutException, NoSuchElementException):
        logging.warning(
            f"爬取 {stock_id} ({url}) 失敗 (Timeout 或 找不到元素)。網頁可能改版或載入過久。")
        return None, None
    except WebDriverException as e:
        logging.error(f"爬取 {stock_id} ({url}) 時 Selenium Driver 發生錯誤: {e}")
        return None, None
    except (IndexError, KeyError, AttributeError) as e:
        logging.warning(
            f"爬取 {stock_id} ({url}) 成功，但解析資料失敗 (Index/Key/AttributeError)。網頁格式可能已變動: {e}")
        return None, None
    except Exception as e:
        logging.error(f"爬取 {stock_id} ({url}) 過程中發生未預期錯誤: {e}")
        return None, None
    finally:
        if driver:
            driver.quit()

# doc: =============================================================================
# doc: (爬蟲核心 2/3) 每月投信前十大持股
# doc: =============================================================================


def crawl_sitca_monthly_top10(year: int, month: int, session: requests.Session) -> pd.DataFrame:
    """
    doc: 爬取 SITCA 網站指定年月的「基金月前十大投資明細」。
    [修改] 增加 requests.exceptions.RequestException 錯誤處理。
    """
    url = "https://www.sitca.org.tw/ROC/Industry/IN2629.aspx?pid=IN22601_04"
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

        # [修改] 增加 timeout
        res_get = session.get(url, headers=headers, timeout=30)
        res_get.raise_for_status()  # 檢查 HTTP 錯誤

        soup = BeautifulSoup(res_get.text, 'html.parser')
        viewstate = soup.find('input', {'id': '__VIEWSTATE'}).get('value')
        eventvalidation = soup.find(
            'input', {'id': '__EVENTVALIDATION'}).get('value')
        viewstategenerator = soup.find(
            'input', {'id': '__VIEWSTATEGENERATOR'}).get('value')

        payload = {
            '__EVENTTARGET': '', '__EVENTARGUMENT': '', '__LASTFOCUS': '',
            '__VIEWSTATE': viewstate, '__VIEWSTATEGENERATOR': viewstategenerator,
            '__EVENTVALIDATION': eventvalidation,
            'ctl00$ContentPlaceHolder1$ddlQ_YM': f"{year}{month:02d}",
            'ctl00$ContentPlaceHolder1$rdo1': 'rbClass',
            'ctl00$ContentPlaceHolder1$ddlQ_Class': 'AA1',
            'ctl00$ContentPlaceHolder1$BtnQuery': '查詢',
        }

        # [修改] 增加 timeout
        res_post = session.post(url, data=payload, headers=headers, timeout=90)
        res_post.raise_for_status()

        soup = BeautifulSoup(res_post.text, 'html.parser')
        header_cell = soup.find('td', class_='DTHeader')
        if not header_cell or '基金名稱' not in header_cell.get_text():
            # [修改] print 改為 logging.info
            logging.info(f"-> [月報] {year} 年 {month} 月查無資料。")
            return pd.DataFrame()

        table = header_cell.find_parent('tr').find_parent('table')
        all_rows_data = []
        current_fund_name = ""
        for row in table.find_all('tr'):
            if row.find('td', class_='DTHeader') or row.find('td', class_='DTsubtotal'):
                continue
            cells = row.find_all('td')
            if not cells:
                continue

            if 'rowspan' in cells[0].attrs:
                current_fund_name = cells[0].text.strip()
                row_data = [current_fund_name] + [cell.text.strip()
                                                  for cell in cells[1:]]
            else:
                row_data = [current_fund_name] + [cell.text.strip()
                                                  for cell in cells]

            if len(row_data) >= 10:
                selected_data = {
                    '基金名稱': row_data[0], '名次': row_data[1], '標的種類': row_data[2],
                    '標的代號': row_data[3], '標的名稱': row_data[4], '金額': row_data[5],
                    '占基金淨資產價值之比例(%)': row_data[9]
                }
                if re.match(r'^\\d{4}$', selected_data['標的代號']):
                    all_rows_data.append(selected_data)

        return pd.DataFrame(all_rows_data)

    # [修改] 增加錯誤處理
    except requests.exceptions.RequestException as e:
        logging.warning(f"-> [月報] {year} 年 {month} 月發生網路錯誤: {e}")
        return pd.DataFrame()
    except (AttributeError, IndexError, KeyError) as e:
        logging.warning(f"-> [月報] {year} 年 {month} 月解析資料失敗 (網頁格式可能變動): {e}")
        return pd.DataFrame()
    except Exception as e:
        logging.error(f"-> [月報] {year} 年 {month} 月發生未預期錯誤: {e}")
        return pd.DataFrame()

# doc: =============================================================================
# doc: (爬蟲核心 3/3) 每季投信1%以上持股
# doc: =============================================================================


def crawl_sitca_quarterly_over_1_percent(year: int, month: int, session: requests.Session) -> pd.DataFrame:
    """
    doc: 爬取 SITCA 網站指定年季的「占基金淨資產價值1%以上投資明細」。
    [修改] 增加 requests.exceptions.RequestException 錯誤處理。
    """
    url = "https://www.sitca.org.tw/ROC/Industry/IN2630.aspx?pid=IN22601_05"
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

        # [修改] 增加 timeout
        res_get = session.get(url, headers=headers, timeout=30)
        res_get.raise_for_status()
        soup = BeautifulSoup(res_get.text, 'html.parser')
        viewstate = soup.find('input', {'id': '__VIEWSTATE'}).get('value')
        eventvalidation = soup.find(
            'input', {'id': '__EVENTVALIDATION'}).get('value')
        viewstategenerator = soup.find(
            'input', {'id': '__VIEWSTATEGENERATOR'}).get('value')

        payload = {
            '__EVENTTARGET': '', '__EVENTARGUMENT': '', '__LASTFOCUS': '',
            '__VIEWSTATE': viewstate, '__VIEWSTATEGENERATOR': viewstategenerator,
            '__EVENTVALIDATION': eventvalidation,
            'ctl00$ContentPlaceHolder1$ddlQ_YM': f"{year}{month:02d}",
            'ctl00$ContentPlaceHolder1$rdo1': 'rbClass',
            'ctl00$ContentPlaceHolder1$ddlQ_Class': 'AA1',
            'ctl00$ContentPlaceHolder1$BtnQuery': '查詢',
        }

        # [修改] 增加 timeout
        res_post = session.post(url, data=payload, headers=headers, timeout=90)
        res_post.raise_for_status()

        soup = BeautifulSoup(res_post.text, 'html.parser')
        header_cell = soup.find('td', class_='DTHeader')
        if not header_cell or '基金名稱' not in header_cell.get_text():
            # [修改] print 改為 logging.info
            logging.info(f"-> [季報] {year} 年 {month} 月查無資料。")
            return pd.DataFrame()

        # ... (以下解析邏輯與您原檔相同) ...
        table = header_cell.find_parent('tr').find_parent('table')
        all_rows_data = []
        current_fund_name = ""
        for row in table.find_all('tr'):
            if row.find('td', class_='DTHeader') or row.find('td', class_='DTsubtotal'):
                continue
            cells = row.find_all('td')
            if not cells:
                continue

            if 'rowspan' in cells[0].attrs:
                current_fund_name = cells[0].text.strip()
                row_data = [current_fund_name] + [cell.text.strip()
                                                  for cell in cells[1:]]
            else:
                row_data = [current_fund_name] + [cell.text.strip()
                                                  for cell in cells]

            stock_id = row_data[2].strip() if len(row_data) > 2 else ""
            if len(row_data) >= 9 and re.match(r'^\\d{4}$', stock_id):
                selected_data = {
                    '基金名稱': row_data[0], '標的種類': row_data[1], '標的代號': stock_id,
                    '標的名稱': row_data[3], '金額': row_data[4],
                    '占基金淨資產價值之比例(%)': row_data[8]
                }
                all_rows_data.append(selected_data)

        return pd.DataFrame(all_rows_data)

    # [修改] 增加錯誤處理
    except requests.exceptions.RequestException as e:
        logging.warning(f"-> [季報] {year} 年 {month} 月發生網路錯誤: {e}")
        return pd.DataFrame()
    except (AttributeError, IndexError, KeyError) as e:
        logging.warning(f"-> [季報] {year} 年 {month} 月解析資料失敗 (網頁格式可能變動): {e}")
        return pd.DataFrame()
    except Exception as e:
        logging.error(f"-> [季報] {year} 年 {month} 月發生未預期錯誤: {e}")
        return pd.DataFrame()

# doc: =============================================================================
# doc: (更新流程 1/3) 每日主動式 ETF
# doc: =============================================================================


def update_daily_etf_data():
    """
    doc: 【已並行化】執行每日主動式 ETF 的檢查與更新流程，存入 all_etf_holdings.csv
    """
    print("\n" + "="*60)
    print("--- (1/3) 開始更新 [每日] 主動式 ETF 持股資料 ---")
    print("="*60)

    etf_list = ['00981A', '00982A', '00980A', '00984A']
    csv_filename = "all_etf_holdings.csv"

    if os.path.exists(csv_filename):
        print(f"找到已存在的檔案: {csv_filename}，將會進行增量更新。")
        existing_df = pd.read_csv(csv_filename)
    else:
        print(f"未找到 {csv_filename}，將會建立新檔案。")
        existing_df = pd.DataFrame()

    newly_scraped_data = []

    # --- [修改] START: 改用 ThreadPoolExecutor ---\n",
    print(f"啟動並行爬蟲，共 {len(etf_list)} 支 ETF...")

    # 使用 ThreadPoolExecutor 來並行抓取
    # max_workers = 4 (len(etf_list))，代表 4 個瀏覽器會同時開啟
    with ThreadPoolExecutor(max_workers=len(etf_list)) as executor:
        # 建立一個 future -> etf_id 的字典，以便稍後取回結果時知道是哪一支
        # executor.submit(fn, *args) 會立即提交任務並返回一個 future 物件
        futures = {
            executor.submit(get_etf_holdings_selenium, etf_id): etf_id
            for etf_id in etf_list
        }

        # as_completed 會在任何一個任務 (future) 完成時立即返回
        for future in as_completed(futures):
            etf_id = futures[future]  # 從字典中找回
            print(f"\n--- 處理 ETF: {etf_id} 的爬取結果 ---")

            try:
                # .result() 會獲取任務的返回值 (df, date)
                holdings_df, data_date = future.result()

                # [修改] 檢查 holdings_df 是否有效 (非 None 且非空)
                if holdings_df is not None and not holdings_df.empty and data_date:
                    is_existing = False
                    if not existing_df.empty:
                        # 檢查同樣ETF和同樣日期的資料是否已存在
                        if not existing_df[(existing_df['etf'] == etf_id) & (existing_df['日期'] == data_date)].empty:
                            is_existing = True

                    if is_existing:
                        print(f"-> 資料已是最新 (日期: {data_date})，跳過 {etf_id}")
                    else:
                        print(f"-> 發現新資料 (日期: {data_date})，準備寫入 {etf_id}")
                        holdings_df['etf'] = etf_id
                        holdings_df['日期'] = data_date
                        newly_scraped_data.append(holdings_df)
                else:
                    # [修改] 將原來的 print 改為 logging.warning
                    logging.warning(f"-> 未能成功抓取到 {etf_id} 的持股資料或日期。")

            except Exception as e:
                # 確保即使單一執行緒出錯，也不會讓整個程式崩潰
                logging.error(f"-> 抓取 {etf_id} 過程中執行緒發生嚴重錯誤: {e}")

    # --- [修改] END: ThreadPoolExecutor 區塊 ---\n",

    if newly_scraped_data:
        print("\n正在合併新舊 ETF 資料...")
        new_df = pd.concat(newly_scraped_data, ignore_index=True)
        combined_df = pd.concat([existing_df, new_df], ignore_index=True)

        final_columns = ['日期', 'etf', '代號', '名稱', '權重', '持有數', '單位']

        # [修改] 確保 final_columns 只包含 combined_df 中實際存在的欄位
        final_columns = [
            col for col in final_columns if col in combined_df.columns]
        combined_df = combined_df[final_columns]

        combined_df.drop_duplicates(
            subset=['日期', 'etf', '代號'], keep='last', inplace=True)
        combined_df.to_csv(csv_filename, index=False, encoding='utf-8-sig')
        print(f"[每日] ETF 資料已成功更新並儲存至 {csv_filename}")
    else:
        print("\n本次執行未抓取到任何新的 [每日] ETF 資料。")

# doc: =============================================================================
# doc: (更新流程 2/3) 每月投信前十大持股
# doc: =============================================================================


def update_monthly_data(session: requests.Session):
    """
    doc: 【已並行化】執行每月持股資料的檢查與更新流程，存入 sitca_fund_holdings.csv
    """
    print("\n" + "="*60)
    print("--- (2/3) 開始更新 [每月] 前十大持股資料 ---")
    print("="*60)

    csv_filename = 'sitca_fund_holdings.csv'
    df_existing = pd.DataFrame()

    if os.path.exists(csv_filename):
        print(f"發現已存在檔案 '{csv_filename}'，將從上次結束的地方繼續更新。")
        df_existing = pd.read_csv(csv_filename)
        if not df_existing.empty:
            df_existing['日期'] = pd.to_datetime(df_existing['日期'])
            start_date = df_existing['日期'].max() + pd.DateOffset(months=1)
        else:
            start_date = pd.to_datetime('2015-06-01')
    else:
        print(f"未發現舊檔案，將從 2015年6月 開始全新抓取。")
        start_date = pd.to_datetime('2015-06-01')

    today = pd.Timestamp.now()
    end_date = today - pd.DateOffset(months=1 if today.day > 10 else 2)
    date_range = pd.date_range(start_date, end_date, freq='MS')

    if date_range.empty:
        print("您的 [每月] 資料已經是最新，無需更新！")
        return

    print(
        f"準備爬取從 {date_range[0].strftime('%Y-%m')} 到 {date_range[-1].strftime('%Y-%m')} 的 [每月] 資料...")
    print(f"將使用 {MAX_WORKERS_SITCA} 個並行 worker...")

    all_new_dataframes = [df_existing] if not df_existing.empty else []

    # --- [修改] START: 改用 ThreadPoolExecutor ---\n",
    with ThreadPoolExecutor(max_workers=MAX_WORKERS_SITCA) as executor:
        # 建立 future -> date 的字典
        futures = {
            executor.submit(crawl_sitca_monthly_top10, date.year, date.month, session): date
            for date in date_range
        }

        for future in as_completed(futures):
            date = futures[future]  # 獲取對應的日期
            try:
                df_month = future.result()  # 獲取爬蟲結果
                if not df_month.empty:
                    print(f"-> 成功抓取 [月報] {date.strftime('%Y-%m')} 資料")
                    df_month['日期'] = date
                    all_new_dataframes.append(df_month)
                # (原版的 '查無資料' 訊息已在 crawl_... 函式中印出)

            except Exception as e:
                print(f"-> 抓取 [月報] {date.strftime('%Y-%m')} 過程中執行緒發生嚴重錯誤: {e}")

    # --- [修改] END: ThreadPoolExecutor 區塊 ---\n",

    # 檢查是否有 *新* 資料被加入 (原邏輯是 > 0，若已有舊資料，應 > 1)
    if len(all_new_dataframes) > (1 if not df_existing.empty else 0):
        full_df = pd.concat(all_new_dataframes, ignore_index=True)
        numeric_cols = ['名次', '金額', '占基金淨資產價值之比例(%)']
        for col in numeric_cols:
            full_df[col] = pd.to_numeric(full_df[col].astype(
                str).str.replace(',', ''), errors='coerce')

        full_df['日期'] = pd.to_datetime(full_df['日期'])
        full_df.drop_duplicates(inplace=True)
        full_df.to_csv(csv_filename, index=False, encoding='utf-8-sig')
        print(f"\n[每月] 資料更新完畢，已全部儲存至 {csv_filename}")
    else:
        print("\n本次執行未抓取到新的 [每月] 資料。")


# doc: =============================================================================
# doc: (更新流程 3/3) 每季投信1%以上持股
# doc: =============================================================================
def update_quarterly_data(session: requests.Session):
    """
    doc: 【已並行化】執行每季持股資料的檢查與更新流程，存入 sitca_fund_holdings_over_1_percent_quarterly.csv
    """
    print("\n" + "="*60)
    print("--- (3/3) 開始更新 [每季] 1%以上持股資料 ---")
    print("="*60)

    csv_filename = 'sitca_fund_holdings_over_1_percent_quarterly.csv'
    df_existing = pd.DataFrame()

    if os.path.exists(csv_filename):
        print(f"發現已存在檔案 '{csv_filename}'，將從上次結束的地方繼續更新。")
        df_existing = pd.read_csv(csv_filename)
        if not df_existing.empty:
            df_existing['日期'] = pd.to_datetime(df_existing['日期'])
            start_date = df_existing['日期'].max() + pd.DateOffset(months=3)
        else:
            start_date = pd.to_datetime('2015-06-01')
    else:
        print(f"未發現舊檔案，將從 2015年6月 開始全新抓取。")
        start_date = pd.to_datetime('2015-06-01')

    end_date = pd.Timestamp.now()
    # 修正: 'QE' 是季末 (Quarter End)，SITCA 資料是以 3, 6, 9, 12 月為準
    # 例如 2015-06-01 (start) -> 2015-06-30 (QE)
    all_possible_dates = pd.date_range(start_date, end_date, freq='Q')

    # 季報通常在下一個月 10 號左右公布
    # 直接在季末日期上加上天數 (例如 10 天或 11 天)
    crawlable_dates = [d for d in all_possible_dates if pd.Timestamp.now() > (
        d + pd.DateOffset(days=11))]  # 建議用 11 天 (10號公布，11號抓) 較保險

    if not crawlable_dates:
        print("您的 [每季] 資料已經是最新，或最新一季資料尚未公佈，無需更新！")
        return

    print(
        f"準備爬取從 {crawlable_dates[0].strftime('%Y-%m')} 到 {crawlable_dates[-1].strftime('%Y-%m')} 的 [每季] 資料...")
    print(f"將使用 {MAX_WORKERS_SITCA} 個並行 worker...")

    all_new_dataframes = [df_existing] if not df_existing.empty else []

    # --- [修改] START: 改用 ThreadPoolExecutor ---\n",
    with ThreadPoolExecutor(max_workers=MAX_WORKERS_SITCA) as executor:
        # 建立 future -> date 的字典
        futures = {
            executor.submit(crawl_sitca_quarterly_over_1_percent, date.year, date.month, session): date
            for date in crawlable_dates
        }

        for future in as_completed(futures):
            date = futures[future]  # 獲取對應的日期
            try:
                df_quarter = future.result()  # 獲取爬蟲結果
                if not df_quarter.empty:
                    print(f"-> 成功抓取 [季報] {date.strftime('%Y-%m')} 資料")
                    df_quarter['日期'] = date
                    all_new_dataframes.append(df_quarter)

            except Exception as e:
                print(f"-> 抓取 [季報] {date.strftime('%Y-%m')} 過程中執行緒發生嚴重錯誤: {e}")

    # --- [修改] END: ThreadPoolExecutor 區塊 ---\n",

    if len(all_new_dataframes) > (1 if not df_existing.empty else 0):
        full_df = pd.concat(all_new_dataframes, ignore_index=True)
        numeric_cols = ['金額', '占基金淨資產價值之比例(%)']
        for col in numeric_cols:
            full_df[col] = pd.to_numeric(full_df[col].astype(
                str).str.replace(',', ''), errors='coerce')

        full_df['日期'] = pd.to_datetime(full_df['日期'])
        full_df.drop_duplicates(inplace=True)
        full_df.to_csv(csv_filename, index=False, encoding='utf-8-sig')
        print(f"\n[每季] 資料更新完畢，已全部儲存至 {csv_filename}")
    else:
        print("\n本次執行未抓取到新的 [每季] 資料。")


# doc: =============================================================================
# doc: 主執行函式
# doc: =============================================================================
def main():
    """
    doc: 主執行函式，依序更新每日、每月與每季資料。
    """
    # 執行每日 ETF 更新 (使用 Selenium，已並行化)
    update_daily_etf_data()

    # 針對 SITCA 的更新，使用同一個 session 物件 (thread-safe)
    with requests.Session() as session:
        # 執行每月 SITCA 更新 (使用 Requests，已並行化)
        update_monthly_data(session)
        # 執行每季 SITCA 更新 (使用 Requests，已並行化)
        update_quarterly_data(session)

    print("\n" + "="*60)
    print("--- 所有更新任務已完成 ---")
    print("="*60)


if __name__ == "__main__":
    main()


--- (1/3) 開始更新 [每日] 主動式 ETF 持股資料 ---
未找到 all_etf_holdings.csv，將會建立新檔案。
啟動並行爬蟲，共 4 支 ETF...

--- 處理 ETF: 00982A 的爬取結果 ---
-> 發現新資料 (日期: 2026/04/02)，準備寫入 00982A

--- 處理 ETF: 00980A 的爬取結果 ---
-> 發現新資料 (日期: 2026/04/02)，準備寫入 00980A

--- 處理 ETF: 00981A 的爬取結果 ---
-> 發現新資料 (日期: 2026/04/02)，準備寫入 00981A

--- 處理 ETF: 00984A 的爬取結果 ---
-> 發現新資料 (日期: 2026/04/02)，準備寫入 00984A

正在合併新舊 ETF 資料...
[每日] ETF 資料已成功更新並儲存至 all_etf_holdings.csv

--- (2/3) 開始更新 [每月] 前十大持股資料 ---
未發現舊檔案，將從 2015年6月 開始全新抓取。
準備爬取從 2015-06 到 2026-02 的 [每月] 資料...
將使用 8 個並行 worker...
